# Density-Based Clustering: DBSCAN

Computer Science Dept, Ferdowsi University of Mashhad

## Where this page sits

The [families page](Three-Families-of-Clustering-Algorithms.ipynb) sorts
clustering methods by what they assume about the data. K-means and K-medoids
belong to the *direct partitioning* family: they assume the number of clusters
$K$ is known, and they assume a cluster is the set of points closest to one
centre. Both assumptions are visible in the result. Voronoi cells are convex,
so a crescent-shaped cluster is cut in two, and every point is assigned
somewhere, even the point that sits alone in a corner and clearly belongs to
nothing.

DBSCAN (Ester, Kriegel, Sander and Xu, 1996) belongs to the *mode seeking*
family. It assumes instead that

* a cluster is a **region of high density** surrounded by regions of low
  density,
* the number of clusters need not be given - it is whatever the density
  structure contains,
* and a point in a low-density region is **noise**, not a member of anything.

### What you should be able to do at the end

* state the definitions of a core point, density-reachability and
  density-connectivity;
* write DBSCAN from scratch and explain the role of `eps` and `min_samples`;
* draw the two moons and say exactly why K-means fails on them and DBSCAN does
  not;
* choose `eps` from a $k$-distance plot and read the sensitivity of the result
  to that choice;
* name the two limitations that later algorithms (OPTICS, HDBSCAN) repair.

## 1. Density, in three definitions

Fix two parameters: a radius $\varepsilon$ (`eps`) and an integer
$\mathrm{MinPts}$ (`min_samples`). The $\varepsilon$-neighbourhood of a point
$p$ is

$$
N_{\varepsilon}(p) = \{\, q \in D \;:\; d(p, q) \le \varepsilon \,\}.
$$

Now the vocabulary has four levels:

* **core point** - a point with $|N_{\varepsilon}(p)| \ge \mathrm{MinPts}$.
  There is enough data around it for a cluster to continue.
* **directly density-reachable** - $q$ is directly density-reachable from $p$
  if $q \in N_{\varepsilon}(p)$ and $p$ is a core point.
* **density-reachable** - the transitive closure of the previous relation: a
  chain $p_1, \dots, p_n$ with each $p_{i+1}$ directly density-reachable from
  $p_i$.
* **density-connected** - $p$ and $q$ are density-connected if some point $o$
  exists from which both are density-reachable.

A **cluster** is a maximal set of density-connected points, and **noise** is
what is left over. The asymmetry in the second definition is the whole trick:
reachability runs *outward from core points*, so a sparse point at the edge of
a cluster joins it (it is one of the core point's neighbours) but cannot
extend it, and a sparse region cannot chain two dense regions together.

Two parameters leave one degree of freedom: `min_samples` is usually taken as
the dimension plus one, or about twice the dimension, and `eps` is then read
off the data. Section 4 does that.

Point types in a picture: **core** points (dense inside), **border** points
(dense enough to belong, not dense enough to be core) and **noise** (neither).

## 2. The algorithm

DBSCAN does not iterate: it sweeps the points once and grows a cluster from
every core point it has not seen yet.

```
for every unvisited point p:
    if p is not a core point:
        mark p as noise (it may be reclaimed later as a border point)
    else:
        start a new cluster C
        seeds = N_eps(p)
        while seeds is not empty:
            take q from seeds
            if q was noise: mark q as C        # a border point
            if q is not yet visited:
                mark q as C
                if q is a core point: seeds += N_eps(q)
```

The inner loop is a breadth-first search over the density-reachable set, so a
cluster is discovered as one connected component of the "is a neighbour of a
core point" graph. The important asymmetry from Section 1 lives in the last
two lines: only a **core** point adds its neighbours to the frontier.

The cost is dominated by the neighbourhood queries, $N$ of them. With a
spatial index (a $k$-d tree or a ball tree) each query is about
$O(\log N)$ in low dimension, which gives $O(N \log N)$ overall; without an
index, or when the dimension is high and the index degenerates, each query
scans the data and the cost is $O(N^2)$. The implementation below computes the
full distance matrix for clarity, which is the $O(N^2)$ variant - honest for a
few hundred points, wrong for a million.

The result is not unique in one detail: a point can be a border point of two
clusters, and which one claims it depends on the order of the scan. Core
points always end up in the same cluster.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import DBSCAN as SklearnDBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler


def dbscan(X, eps, min_samples):
    """The algorithm above, with the full distance matrix as the index."""
    n = len(X)
    distance = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(-1))
    neighbours = [np.where(distance[i] <= eps)[0] for i in range(n)]

    UNVISITED, NOISE = -2, -1
    labels = np.full(n, UNVISITED)
    n_clusters = 0

    for p in range(n):
        if labels[p] != UNVISITED:
            continue
        if len(neighbours[p]) < min_samples:
            labels[p] = NOISE          # provisional; a later cluster may claim it
            continue

        labels[p] = n_clusters
        seeds = list(neighbours[p])
        while seeds:
            q = seeds.pop()
            if labels[q] == NOISE:     # border point of this cluster
                labels[q] = n_clusters
            if labels[q] != UNVISITED:
                continue
            labels[q] = n_clusters
            if len(neighbours[q]) >= min_samples:   # only core points expand
                seeds.extend(neighbours[q])
        n_clusters += 1

    return labels


def describe(labels):
    clusters = [k for k in np.unique(labels) if k != -1]
    sizes = [int((labels == k).sum()) for k in clusters]
    return f"{len(clusters)} cluster(s) {sizes}, {int((labels == -1).sum())} noise point(s)"


## 3. Two moons: where K-means breaks

The standard counterexample is two interleaving crescents. K-means minimises
the squared distance to two centres, so it can only cut the plane with a
straight line; the best straight cut runs *along* the crescents and splits each
one. DBSCAN does not know about centres. It only asks whether a point has
neighbours, and inside a crescent every point does.

In [ ]:
X, y = make_moons(n_samples=400, noise=0.06, random_state=1)
X = StandardScaler().fit_transform(X)

kmeans_labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)
dbscan_labels = dbscan(X, eps=0.3, min_samples=5)
sklearn_labels = SklearnDBSCAN(eps=0.3, min_samples=5).fit_predict(X)

print("K-means          :", describe(kmeans_labels),
      f"(ARI {adjusted_rand_score(y, kmeans_labels):.3f})")
print("DBSCAN (ours)    :", describe(dbscan_labels),
      f"(ARI {adjusted_rand_score(y, dbscan_labels):.3f})")
print("DBSCAN (sklearn) :", describe(sklearn_labels),
      f"(ARI {adjusted_rand_score(y, sklearn_labels):.3f})")
print("ours and sklearn agree on every point:",
      bool(np.array_equal(dbscan_labels, sklearn_labels)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, labels, title in [(axes[0], kmeans_labels, "K-means, K = 2"),
                          (axes[1], dbscan_labels, "DBSCAN, eps = 0.3, min_samples = 5")]:
    noise = labels == -1
    ax.scatter(X[~noise, 0], X[~noise, 1], c=labels[~noise], cmap="viridis", s=18)
    ax.scatter(X[noise, 0], X[noise, 1], c="crimson", marker="x", s=30, label="noise")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    if noise.any():
        ax.legend(loc="upper right")
fig.suptitle("Two moons: a straight cut cannot follow a crescent")
plt.show()

The left panel is the failure: each crescent is half blue and half yellow,
because the boundary between the two Voronoi cells is a line. The right panel
recovers both crescents exactly, and it was not told that there are two - the
number of clusters fell out of the density.

The printed comparison is a check on the implementation rather than a result:
our labels should match `sklearn.cluster.DBSCAN` point for point, up to the
order in which clusters are numbered. (If the labels differ, it is worth
finding out whether it is the numbering or a border point.)

## 4. Choosing `eps`: the $k$-distance plot

`min_samples` is the easier parameter: it is a statement about how many
neighbours a point needs before its region counts as dense, and it scales with
the dimension. `eps` is harder, because it is a length and must be compared
with the actual spacing of the data.

The usual diagnostic fixes $k = \mathrm{MinPts}$ and, for every point, computes
the distance to its $k$-th nearest neighbour. Sort those distances in
increasing order and plot them. Points inside clusters have small $k$-distances
and form a slowly rising line; noise has a much larger one. The **knee** of the
curve is the value at which "close" turns into "far", and is the natural
`eps`.

In [ ]:
from sklearn.neighbors import NearestNeighbors

k = 5
k_distances = np.sort(
    NearestNeighbors(n_neighbors=k).fit(X).kneighbors(X)[0][:, -1]
)

eps_values = [0.15, 0.2, 0.25, 0.3, 0.4, 0.6, 0.7]
print(f"{'eps':>6}  {'clusters':>8}  {'noise':>6}  {'ARI':>6}")
for eps in eps_values:
    labels = dbscan(X, eps=eps, min_samples=k)
    n_clusters = len([c for c in np.unique(labels) if c != -1])
    print(f"{eps:6.2f}  {n_clusters:8d}  {int((labels == -1).sum()):6d}"
          f"  {adjusted_rand_score(y, labels):6.3f}")

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(k_distances, marker=".", markersize=3, linewidth=0.8)
ax.axhline(0.3, color="crimson", linestyle="--", label="eps = 0.3")
ax.set_xlabel(f"points sorted by distance to their {k}th neighbour")
ax.set_ylabel(f"{k}-distance")
ax.set_title("The knee of the k-distance curve suggests eps")
ax.legend()
plt.show()

The curve bends around $0.15$-$0.2$ and then climbs steeply: everything to the
right of the knee is the sparse region. The table is the more useful half of
the figure, because it shows the shape of the dependence rather than one value:

* too small (`eps` $=0.15$) - the clusters fragment, few points have enough
  neighbours;
* at the knee (`eps` $=0.2$) - three pieces, one crescent plus a split one;
* just above the knee (`eps` $=0.25$ to $0.6$) - the two crescents, and the
  answer does not change across that whole range, which is the desirable half
  of the story: near the knee the result is stable;
* too large (`eps` $=0.7$) - the gap between the crescents is bridged and the
  two clusters merge into one.

That is the honest summary of DBSCAN's main weakness: it has one density
threshold for the whole data set, and a single threshold is a strong assumption.

## 5. When the density is not uniform

Real data rarely has one density. The experiment below puts three clouds side
by side - 500 points with standard deviation $0.15$, 150 with $0.7$, and 50 with
$1.8$ - and runs DBSCAN at two values of `eps`.

* at `eps` $=0.12$, small enough for the tight cloud, the medium cloud
  shatters into three tiny fragments and the loose cloud is entirely noise;
* at `eps` $=0.8$, large enough to see the loose cloud, the density contrast
  is gone: 664 of the 700 points fall into a single cluster, the rest are
  noise or a small leftover fragment.

Values in between do no better: at `eps` $=0.3$ the tight cloud is found and
the middle one is recovered minus a quarter of its points, while all but four
of the loose points are noise; at `eps` $=0.6$ the middle cloud is intact and
the loose points are being absorbed into it. No value of `eps` returns all
three clouds. That is not a tuning failure, it is the single-threshold
model.

In [ ]:
# one tight cloud, one medium cloud and one loose cloud, left to right
X2, _ = make_blobs(n_samples=[500, 150, 50],
                   centers=[[-2.0, 0.0], [0.5, 0.0], [3.0, 0.0]],
                   cluster_std=[0.15, 0.7, 1.8], random_state=11)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, eps in zip(axes, [0.12, 0.8]):
    labels = dbscan(X2, eps=eps, min_samples=5)
    noise = labels == -1
    ax.scatter(X2[~noise, 0], X2[~noise, 1], c=labels[~noise], cmap="viridis", s=18)
    ax.scatter(X2[noise, 0], X2[noise, 1], c="crimson", marker="x", s=30, label="noise")
    ax.set_title(f"eps = {eps}: {describe(labels)}")
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(loc="upper right")
plt.show()

Both panels are wrong in a different way, and the two failure modes
correspond to two later algorithms:

* **OPTICS** (Ankerst et al., 1999) replaces the single radius by an ordering
  of the points by reachability, from which clusters at several densities can
  be read off, instead of committing to one `eps`;
* **HDBSCAN** (Campello, Moulavi and Sander, 2013) builds the density
  hierarchy over all values of `eps` and extracts the most persistent clusters,
  which is what `sklearn.cluster.HDBSCAN` implements today.

Two more limits are worth stating before the summary. First, "density" is
computed with a distance, and in high dimensions distances concentrate (see
[the curse of dimensionality](High-Dimensional-Data-and-The-Curse-of-Dimensionality.ipynb)),
so the contrast between dense and sparse fades and DBSCAN degenerates towards
"everything is one cluster" or "everything is noise" unless the data is first
reduced. Second, because clusters are grown from core points and border points
are claimed by whichever cluster finds them first, labels are not a
deterministic function of the data alone - only core points are.

## Summary

* DBSCAN defines a cluster as a maximal density-connected set, with density
  measured by an $\varepsilon$-ball containing at least `MinPts` points.
* Core points are the only points that propagate a cluster; border points join
  it; the rest is noise. That is how arbitrarily shaped clusters are found and
  how outliers are refused rather than absorbed.
* The number of clusters is an output, not an input.
* The algorithm is a single sweep with neighbourhood queries: $O(N \log N)$
  with a spatial index, $O(N^2)$ without.
* One global density threshold is the limitation; OPTICS, HDBSCAN and
  dimension reduction are the usual repairs.

## Exercises

1. Modify `dbscan` to also return the point types (core, border, noise) and
   colour the two-moons figure by type. Which points are at risk of switching
   clusters when the processing order changes?
2. Shuffle the rows of `X` before clustering and rerun. Which labels change?
   Check whether any *core* point changes cluster.
3. Sweep `min_samples` from 2 to 20 at `eps = 0.3` on the moons. Why does a
   larger `min_samples` eventually lose the thin ends of the crescents?
4. Replace the $k$-distance plot with the sorted distance to the nearest
   *non*-neighbour. Is the knee easier to see?
5. Run DBSCAN on the two concentric circles (`make_circles`). Which `eps`
   succeeds where K-means cannot, and what happens to the inner circle when
   `eps` grows?
6. Time the implementation for $N = 200, 400, 800, 1600$. Fit a power law to
   the run times and compare the exponent with the two complexity claims in
   Section 2.
7. Reduce the two-moons data to one dimension with PCA and run DBSCAN. Why
   does it fail, and what does that say about using DBSCAN after an
   unsupervised projection?

## Sources

* Ester, Kriegel, Sander and Xu, "A Density-Based Algorithm for Discovering
  Clusters in Large Spatial Databases with Noise", KDD 1996 {cite}`Ester1996` -
  the definitions, the algorithm and the original experiments.
* The density-based family in the taxonomy of
  {cite}`zaki2020data`, Chapter 13 (DBSCAN, OPTICS and density-based
  clustering).
* `sklearn.cluster.DBSCAN` and `sklearn.cluster.HDBSCAN` documentation for the
  API and for the practical defaults used above.
* OPTICS: Ankerst, Breunig, Kriegel and Sander, SIGMOD 1999. HDBSCAN:
  Campello, Moulavi and Sander, PAKDD 2013.